In [2]:
import os
import numpy as np

from PIL import Image
%load_ext autoreload
%autoreload 2

# Make a dataloader that mixes wevrything randomly loads the images splits a train and test and computes the embeddings
from pathlib import Path

from data_handler import EleHandler
from seek_code import SEEK

import torch
import torch.nn.functional as F
import torchvision.transforms as T
from torch.utils.data import DataLoader
from torch.utils.data.sampler import RandomSampler
import timm


/data/vision/beery/scratch/antoine/micromamba/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def closest_valid_one_hot(prob_vector):
    """
    Converts a probability tensor into the closest valid one-hot encoded representation for each attribute.
    """
    closest_one_hot = []
    index = 0

    for group in SEEK.attribute_names:
        slice_length = SEEK.lengths[group]
        prob_slice = prob_vector[:, index:index + slice_length]
        one_hot_slice = torch.zeros_like(prob_slice)
        
        max_indices = torch.argmax(prob_slice, dim=1)
        one_hot_slice[torch.arange(prob_slice.size(0)), max_indices] = 1.0
        closest_one_hot.append(one_hot_slice)
        
        index += slice_length

    return torch.cat(closest_one_hot, dim=1)

In [4]:
def calculate_attribute_accuracy(predicted, labels):
    # Convert predicted logits to closest valid one-hot representation
    predicted_one_hot = closest_valid_one_hot(predicted)
    
    # Initialize accuracy tracking for all attributes, including individual and averaged
    accuracies = {name: 0 for name in SEEK.attribute_names}

    index = 0
    for name in SEEK.attribute_names:
        length = SEEK.lengths[name]
        pred_slice = predicted_one_hot[:, index:index + length]
        label_slice = labels[:, index:index + length]

        # Calculate per-sample accuracy
        pred_indices = torch.argmax(pred_slice, dim=1)
        label_indices = torch.argmax(label_slice, dim=1)
        accuracies[name] = (pred_indices == label_indices).float().mean().item()

        index += length

    # Include accuracy for whole SEEK code prediction
    correct_whole_code = torch.all(predicted_one_hot == labels, dim=1).float().mean().item()
    accuracies['average'] = sum(accuracies.values()) / len(accuracies)
    accuracies['whole_code'] = correct_whole_code

    return accuracies


In [5]:
def epoch_pass(epoch, loader, training=True):
    model.eval()  # Freeze model for inference
    layer.train() if training else layer.eval()

    total_loss = 0
    num_batches = len(loader)
    epoch_accuracies = {name: 0 for name in SEEK.attribute_names}
    epoch_accuracies['average'] = 0
    epoch_accuracies['whole_code'] = 0

    for batch in loader:
        images, subject_id, ele_id, identified, subject_SEEK, ele_SEEK = batch

        # Prepare labels and transfer them to GPU
        labels = torch.stack([SEEK(s).one_hot_encode() for s in subject_SEEK]).to('cuda')
        images = images.to('cuda')

        # Obtain embeddings from the frozen model
        with torch.no_grad():
            embeddings = model(images)

        # Forward pass through the trainable layer
        outputs = layer(embeddings)
        
        # Calculate loss
        loss = F.mse_loss(outputs, labels)
        total_loss += loss.item()

        # Calculate accuracy for average and whole-code
        predicted_SEEK = closest_valid_one_hot(outputs)
        batch_accuracies = calculate_attribute_accuracy(predicted_SEEK, labels)
        
        for name, value in batch_accuracies.items():            
            epoch_accuracies[name] += value

        # Backpropagation only for the linear layer if training
        if training:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # Calculate epoch-level average loss and accuracies
    average_loss = total_loss / num_batches
    epoch_accuracies = {name: accuracy / num_batches for name, accuracy in epoch_accuracies.items()}
    
    return average_loss, epoch_accuracies


In [6]:
def train_concept_head(train_loader, val_loader, backbone_frozen=True, num_epochs=10):
    """
    Trains a linear layer on top of a pre-trained model to predict SEEK codes.
    """
    for epoch in range(num_epochs):
        train_loss, train_acc = epoch_pass(epoch, train_loader, training=True)
        val_loss, val_acc = epoch_pass(epoch, val_loader, training=False)
        
        if epoch % 5 == 0:
            print(f" Epoch {epoch + 1}/{num_epochs} - Loss : train {train_loss:.4f} val {val_loss:.4f} - Avg accuracy : train {train_acc['average'] * 100:.2f}% val {val_acc['average'] * 100:.2f}% - Whole Code : train {train_acc['whole_code'] * 100:.2f}% val {val_acc['whole_code'] * 100:.2f}%")
    return train_loss, train_acc['average'], train_acc['whole_code'], val_loss, val_acc['average'], val_acc['whole_code']

def test_concept_head(test_loader):

    test_loss, test_acc = epoch_pass(-1, test_loader, training=False)
    
    print("============ Test Summary =========== ")
    print(f"  Test Loss: {test_loss:.4f}")

    # Detailed accuracy for each characteristic
    print("\nDetailed Characteristic Accuracies:")
    for name, value in test_acc.items():
        print(f"  {name} Accuracy: {value * 100:.2f}%")
    
    return test_loss, test_acc['average'], test_acc['whole_code'], test_acc


In [7]:

# Create dataset and dataloaders
dataset = EleHandler( )

train_indices, val_indices, test_indices = dataset.split_along_encounters([0.7,0.15,0.15])

train_loader, val_loader, test_loader = (DataLoader(dataset, batch_size=1024, sampler=RandomSampler(indices), num_workers=4) for indices in [train_indices, val_indices, test_indices])


model = timm.create_model("hf-hub:BVRA/MegaDescriptor-T-224", pretrained=True, num_classes=0) # Mega-Descriptor pretrained
#state_dict = torch.load("../weights/forest_elephants-reid_weights.pt", map_location='cuda') # Pretrained on forest elephants
state_dict = torch.load("../weights/savanna_elephants_md_v2_epoch_60.pt", map_location='cuda') # Pretrained on savannah elephants
model.load_state_dict(state_dict["model"]) if "optimizer" in state_dict else model.load_state_dict(state_dict)
model.to('cuda')

layer = torch.nn.Sequential(
    torch.nn.Linear(768, 63),
    torch.nn.Sigmoid()
).to('cuda')

optimizer = torch.optim.Adam(layer.parameters(), lr=0.001)

train_concept_head(train_loader, val_loader, backbone_frozen = True,  num_epochs=50)
test_concept_head(test_loader)

Train indices: 6108, Validation indices: 1101, Test indices: 1235


/tmp/ipykernel_3981539/78999268.py:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load("../weights/savanna_elephants_md_v2_epoch_60.pt", map_location='

 Epoch 1/50 - Loss : train 0.1933 val 0.1348 - Avg accuracy : train 51.72% val 66.76% - Whole Code : train 0.10% val 0.89%
 Epoch 6/50 - Loss : train 0.1078 val 0.1019 - Avg accuracy : train 69.52% val 71.54% - Whole Code : train 1.79% val 0.78%
 Epoch 11/50 - Loss : train 0.1027 val 0.1034 - Avg accuracy : train 71.15% val 71.32% - Whole Code : train 2.49% val 2.23%
 Epoch 16/50 - Loss : train 0.1000 val 0.1020 - Avg accuracy : train 72.18% val 71.41% - Whole Code : train 3.27% val 2.52%
 Epoch 21/50 - Loss : train 0.0981 val 0.0982 - Avg accuracy : train 72.79% val 72.32% - Whole Code : train 3.72% val 2.71%
 Epoch 26/50 - Loss : train 0.0967 val 0.0965 - Avg accuracy : train 73.31% val 73.52% - Whole Code : train 4.29% val 2.41%
 Epoch 31/50 - Loss : train 0.0955 val 0.0966 - Avg accuracy : train 73.69% val 73.57% - Whole Code : train 4.72% val 5.05%
 Epoch 36/50 - Loss : train 0.0946 val 0.0953 - Avg accuracy : train 74.07% val 74.16% - Whole Code : train 4.93% val 5.75%
 Epoch 41/

(0.09600068628787994,
 0.7370423385873437,
 0.04246658459305763,
 {'sex': 0.9351581335067749,
  'age': 0.9237263202667236,
  'right_tusk': 0.7908258736133575,
  'left_tusk': 0.8346855640411377,
  'R_tear_1': 0.44750167429447174,
  'R_hole_1': 0.669227659702301,
  'R_tear_2': 0.5800573229789734,
  'R_hole_2': 0.7594601809978485,
  'L_tear_1': 0.45683687925338745,
  'L_hole_1': 0.6990567743778229,
  'L_tear_2': 0.5812930464744568,
  'L_hole_2': 0.7775705456733704,
  'right_extreme': 0.8063074052333832,
  'left_extreme': 0.7760478556156158,
  'ear_special': 0.8466773927211761,
  'body_special': 0.908244788646698,
  'average': 0.7370423385873437,
  'whole_code': 0.04246658459305763})

In [ ]:

# Create dataset and dataloaders
dataset = EleHandler( )

train_indices, val_indices, test_indices = dataset.split_along_encounters([0.7,0.15,0.15], hour_delta=2)

train_loader, val_loader, test_loader = (DataLoader(dataset, batch_size=1024, sampler=RandomSampler(indices), num_workers=4) for indices in [train_indices, val_indices, test_indices])


model = timm.create_model("hf-hub:BVRA/MegaDescriptor-T-224", pretrained=True, num_classes=0) # Mega-Descriptor pretrained
#state_dict = torch.load("../weights/forest_elephants-reid_weights.pt", map_location='cuda') # Pretrained on forest elephants
state_dict = torch.load("../weights/savanna_elephants_md_v2_epoch_60.pt", map_location='cuda') # Pretrained on savannah elephants
model.load_state_dict(state_dict["model"]) if "optimizer" in state_dict else model.load_state_dict(state_dict)
model.to('cuda')

layer = torch.nn.Sequential(
    torch.nn.Linear(768, 63),
    torch.nn.Sigmoid()
).to('cuda')

optimizer = torch.optim.Adam(layer.parameters(), lr=0.001)

train_concept_head(train_loader, val_loader, backbone_frozen = True,  num_epochs=50)
test_concept_head(test_loader)